# SD-02 · Back-of-envelope estimation and API styles (REST, GraphQL, gRPC)

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/SD-02.ipynb)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: SD-02
title: "Back-of-envelope estimation and API styles (REST, GraphQL, gRPC)"
track: system-design
date: 2026-10-14
code: ""
status: partial  # not-started | in-progress | partial | done
score: ""
insight: "Estimate first (users x actions / 86,400 x peak, then storage and bandwidth) to see where the design effort belongs; and what you fetch matters more than how you encode it: the same screen was 25 KB in 3 REST calls vs 3 KB in 1 GraphQL call, and protobuf was only about 25% smaller than that JSON."
tags: [SD-02, system-design, estimation, rest, graphql, grpc, pagination, mobile]
pending:
  - "English: 3 sentences with extensible and resilient (past simple vs present perfect)"
  - "Team Lead role lens"
  - "MQ422 in my own words"
  - "Present the casebook in 10 minutes (self-check)"
```

## Goal
Design choices depend on **scale**. An estimate takes five minutes and tells you whether you need one server or fifty, a CDN or none, one database or a cluster. The API style then decides round trips, payload size and how the API can evolve.

# Part 1 · Back-of-envelope estimation

## The recipe
1. **State your assumptions out loud.** An estimate is only as good as its assumptions.
2. **Requests** = users × actions per user per day.
3. **Per second** = requests ÷ 86,400 (about 10⁵ for mental math), then × a **peak factor** (3 to 10: people use apps in the evening).
4. **Storage** = items per day × size per item × days × replicas.
5. **Bandwidth** = requests × response size.

**Units:** capital **B** = bytes (KB, MB, GB: storage). Lowercase **b** = bits (Kb, Mb, Gb: network speed). The factor is 8.

## Worked example: a journaling app, 1,000,000 daily users
Assumptions: 5 reads and 1 write per user per day, an entry is 3 KB, a feed response is 20 KB, peak is 5× the average, 5% of entries carry a 500 KB photo.

| Quantity | Working | Result |
|---|---|---|
| Requests per day | 1M × (5 + 1) | 6,000,000 |
| Average QPS | 6,000,000 ÷ 86,400 | **69** (reads 58, writes 12) |
| Peak QPS (×5) | | **347** (reads 289, writes 58) |
| Text storage | 1M × 3 KB | 3 GB per day, **1.1 TB per year**, 3.3 TB with 3 replicas |
| Photos | 5% × 500 KB = 25 KB per entry on average | 25 GB per day, **9 TB per year** |
| Read traffic | 5M × 20 KB | 100 GB per day, about 9 Mbps average, **46 Mbps peak** |

**What the numbers decide**
- **350 peak QPS:** one modest server can do the work. Run two or three for failure tolerance, not for load.
- **60 peak write QPS:** one Postgres instance, no sharding.
- **1.1 TB per year of text:** fits one database for a year or two.
- **9 TB per year of photos:** **object storage plus a CDN**, not the database. One small assumption (5% photos) made photos dominate.
- **46 Mbps peak:** bandwidth is not the problem. Photos are.

In [ ]:
def estimate(dau, reads, writes, entry_kb, read_kb, peak=5, image_share=0.0, image_kb=0.0):
    seconds = 86_400
    total = dau * (reads + writes)
    result = {
        "requests_per_day": total,
        "avg_qps": total / seconds,
        "peak_qps": total / seconds * peak,
        "text_gb_per_day": dau * writes * entry_kb / 1_000_000,
        "photo_gb_per_day": dau * writes * image_share * image_kb / 1_000_000,
        "read_gb_per_day": dau * reads * read_kb / 1_000_000,
    }
    result["text_gb_per_year"] = result["text_gb_per_day"] * 365
    result["photo_gb_per_year"] = result["photo_gb_per_day"] * 365
    result["read_mbps_avg"] = result["read_gb_per_day"] * 8000 / seconds      # GB/day -> megabits per second
    return result

def show(name, r):
    print(f"--- {name}")
    print(f"avg {r['avg_qps']:.2f} QPS, peak {r['peak_qps']:.2f} QPS")
    print(f"text {r['text_gb_per_day']:.2f} GB/day = {r['text_gb_per_year']:.2f} GB/year (x3 replicas = {r['text_gb_per_year'] * 3:.2f})")
    print(f"photos {r['photo_gb_per_day']:.2f} GB/day = {r['photo_gb_per_year']:.2f} GB/year")
    print(f"read traffic {r['read_gb_per_day']:.1f} GB/day = {r['read_mbps_avg']:.2f} Mbps average, {r['read_mbps_avg'] * 5:.2f} Mbps at peak")

big = estimate(1_000_000, 5, 1, 3, 20, image_share=0.05, image_kb=500)
kaizen = estimate(10_000, 5, 1, 3, 20, image_share=0.05, image_kb=500)
show("1,000,000 daily users", big)
show("Kaizen, 10,000 daily users", kaizen)

assert abs(big["avg_qps"] - 69.44) < 0.01 and abs(big["peak_qps"] - 347.2) < 0.1
assert abs(kaizen["peak_qps"] - 3.47) < 0.01                                  # my own working: 3.47 QPS
assert abs(kaizen["text_gb_per_year"] - 10.95) < 0.01                         # 10.95 GB per year
assert abs(kaizen["photo_gb_per_year"] - 91.25) < 0.01                        # 91.25 GB per year

**What it decides for Kaizen (my own estimate):** about 3.5 peak QPS and 33 GB of text per year (plus 91 GB of photos). One small server and one small database are plenty for years. No CDN, no sharding. **Over-engineering is also a design mistake**, and the estimate is what stops it.

**Scaling check:** 10,000 users is 100× fewer than 1,000,000, so every number should be 100× smaller (347 → 3.47 QPS, 3.3 TB → 33 GB). If it is not, an arithmetic slip is hiding.

## Rules of thumb
| | |
|---|---|
| Seconds in a day | 86,400 ≈ **10⁵** |
| Peak factor | **3 to 10×** the average |
| Replicas | **×3** is a common default |
| 1 million users doing 1 thing a day | about **12 per second** on average |
| Always state | assumptions, units, and peak vs average |

**Latency numbers (rough orders of magnitude; they vary by hardware):**
| Operation | About |
|---|---|
| Read from memory | 100 nanoseconds |
| Read from SSD (random) | 100 microseconds |
| Round trip inside one data centre | 0.5 milliseconds |
| Round trip over a mobile network | **50 to 300 milliseconds** |
| Disk seek (old spinning disk) | 10 milliseconds |

One mobile round trip costs about a hundred times more than a database read, so API design is largely about **reducing round trips**.

## MQ422 · How does a WebSocket server scale to a million phones?
It is an estimation question.
- **Memory:** an open connection keeps a socket plus buffers, assume 10 to 50 KB each: 1,000,000 × 10 to 50 KB = **10 to 50 GB** just to hold the connections.
- **Per server:** tens to hundreds of thousands (limits: file descriptors, memory, kernel). Assume 100,000: **10 servers**, **20 with 2× headroom**.
- **Routing:** the user you want to reach is connected to *some* server. A shared layer (for example Redis pub/sub) finds or notifies it.
- **Reconnect storm:** if a server dies, its 100,000 phones reconnect at once. Spread reconnects with random delay (**jitter**). A million phones reconnecting within 10 seconds is **100,000 new connections per second**.
- **Mobile:** phones lose and regain networks constantly, so tune heartbeats (battery vs detecting dead connections).

# Part 2 · API styles: one screen, three designs
**The screen:** a home feed. It needs the user's name and avatar, the unread-notification count, and 10 entries (id, date, mood, a 100-character snippet, a thumbnail URL).

## REST: resources and URLs
```text
GET /me
GET /notifications/count
GET /entries?limit=10&after=<cursor>
```
- **Good:** every tool understands it (`curl`, browsers, CDNs); `GET` responses are cacheable by HTTP; simple to build and debug.
- **Weak for this screen:** **3 calls**, and `/entries` returns the **whole entry** (2,000-character body, tags, location) when the screen shows a snippet: **over-fetching**.
- **Fixes:** a `fields=` parameter, or a dedicated `GET /home` endpoint (a backend-for-frontend).

## GraphQL: the client describes the shape
One endpoint, `POST /graphql`:
```graphql
query Home {
  me { name avatarUrl }
  unreadCount
  entries(first: 10) { id createdAt mood snippet thumbnailUrl }
}
```
- **Good:** **one round trip** and exactly the fields the screen needs; the app can change its request without a new endpoint. GitHub and Airbnb use it to cut mobile round trips.
- **Cost:** HTTP caching is harder (a POST to one URL); the server needs resolvers, query-cost and depth limits, and care about the **N+1 query** problem; debugging needs more than `curl`.

## gRPC: a typed contract, in binary
```proto
message EntrySummary { string id = 1; int64 created_at = 2; Mood mood = 3; string snippet = 4; string thumbnail_url = 5; }
message HomeFeed { User user = 1; int32 unread_count = 2; repeated EntrySummary entries = 3; }
service Home { rpc GetHomeFeed(HomeFeedRequest) returns (HomeFeed); }
```
- **Good:** compact binary messages (Protocol Buffers) over **HTTP/2**, strong typing with generated Kotlin code, streaming in both directions. Google uses it between internal services. Field **numbers** make evolution safe: never reuse one.
- **Cost:** browsers need a gRPC-web proxy; binary is harder to read while debugging (`grpcurl` helps); no HTTP caching.

## Measured: the same screen, 10 entries with 2,000-character bodies
| Style | Calls | Bytes on the wire | With gzip |
|---|---|---|---|
| REST (resources as they are) | **3** | **25,496** | 6,606 |
| GraphQL (JSON) | **1** | **3,135** | about 1,000 (response) |
| gRPC (protobuf) | **1** | **2,277** | about 940 |

Per entry: 2,512 B (REST) vs 286 B (GraphQL selection) vs 216 B (protobuf).

**The lesson:** selecting only the needed fields cut the size about **8×**. Moving from JSON to binary cut only about another **25%**, and after gzip they are almost equal. **What you send matters much more than how you encode it.** (My sample text is repetitive, so gzip flatters every row; real text compresses less. The protobuf figure came from a compiled `.proto`; the cell below measures REST vs GraphQL with the standard library only. It builds a slightly different sample, so it prints a little different bytes, about 25,075 vs 2,980, but the ratio is the same: about 8× smaller.)

In [ ]:
import gzip, json, random, uuid

random.seed(7)
words = ("today I felt calm after a long walk and wrote down three small things that went well "
         "the morning was slow and quiet and I made tea and read a few pages before work started").split()

def text(n):
    out = []
    while sum(len(w) + 1 for w in out) < n:
        out.append(random.choice(words))
    return " ".join(out)[:n]

def entry(i):
    eid = str(uuid.UUID(int=random.getrandbits(128)))
    body = text(2000)
    return dict(id=eid, userId="u1", createdAt=f"2026-10-{13 - i:02d}T08:00:00Z", updatedAt=f"2026-10-{13 - i:02d}T08:00:00Z",
                mood=random.randint(1, 4), title=text(40), body=body, tags=["gratitude", "work", "sleep"],
                photoUrl=f"https://cdn.example.com/p/{eid}.jpg", thumbnailUrl=f"https://cdn.example.com/t/{eid}.jpg",
                location={"lat": 12.9716, "lon": 77.5946}, wordCount=len(body.split()), visibility="private")

entries = [entry(i) for i in range(10)]
me = dict(id="u1", name="Asha Rao", email="asha.rao@example.com", avatarUrl="https://cdn.example.com/a/u1.jpg",
          createdAt="2025-03-02T10:11:12Z", timezone="Asia/Kolkata", settings={"theme": "dark", "reminders": True})

def j(o): return json.dumps(o, separators=(",", ":")).encode()

rest = [j(me), j({"unread": 3}), j({"items": entries, "nextCursor": "MjAyNi0xMC0wNFQwODowMDowMFo"})]
rest_bytes = sum(map(len, rest))

graphql = j({"data": {"me": {"name": me["name"], "avatarUrl": me["avatarUrl"]}, "unreadCount": 3,
                      "entries": [dict(id=e["id"], createdAt=e["createdAt"], mood=e["mood"],
                                       snippet=e["body"][:100], thumbnailUrl=e["thumbnailUrl"]) for e in entries]}})

print(f"REST    : 3 calls, {rest_bytes:,} bytes, gzip {sum(len(gzip.compress(x)) for x in rest):,}")
print(f"GraphQL : 1 call , {len(graphql):,} bytes, gzip {len(gzip.compress(graphql)):,}")
print(f"selecting only the needed fields made it {rest_bytes / len(graphql):.1f}x smaller")
assert rest_bytes > 5 * len(graphql)

## What it means for latency
Assume a mobile network with a **150 ms round trip** and **5 Mbps**:

| Style | Transfer time | Total time |
|---|---|---|
| REST, 3 calls in **parallel** | 41 ms | about 190 ms |
| REST, 3 calls **one after another** (each needs the previous result) | 41 ms | **about 490 ms** |
| GraphQL, 1 call | 5 ms | about 155 ms |
| gRPC, 1 call | 4 ms | about 154 ms |

On a fast network the payload size matters less than the number of **sequential round trips**. On slow or metered networks, over-fetching also costs battery and money.

## Pagination: cursor vs offset
**Offset** (`?page=3`) means "skip 20 rows, take 10". It breaks on a feed where new items keep arriving, and the database still reads and discards the skipped rows. **Cursor** (`?after=<last item>`) means "items older than this one": stable, and it uses an index.

In [ ]:
feed = [f"E{i}" for i in range(10, 0, -1)]            # newest first: E10 ... E1

def page_offset(items, page, size):
    return items[page * size:(page + 1) * size]

page1 = page_offset(feed, 0, 3)                        # E10, E9, E8
feed = ["E12", "E11"] + feed                           # two new entries arrive

page2_offset = page_offset(feed, 1, 3)                 # skip 3, take 3
idx = feed.index(page1[-1])                            # cursor: "older than the last item I saw"
page2_cursor = feed[idx + 1: idx + 4]

print("page 1        :", page1)
print("page 2, offset:", page2_offset, "  <- E9 and E8 appear AGAIN")
print("page 2, cursor:", page2_cursor)
assert set(page1) & set(page2_offset) == {"E9", "E8"}
assert not set(page1) & set(page2_cursor)

## Which one, when
| Situation | Pick |
|---|---|
| Simple public API, cacheable content, many clients | **REST** |
| Many different screens, mobile clients, deeply related data | **GraphQL** |
| Service-to-service, strict contracts, streaming | **gRPC** |

**Evolving each:** REST adds fields (and versions as a last resort); GraphQL deprecates fields; protobuf never reuses a field number.

**For Kaizen: REST.** It is simple, `curl`-able and cacheable, and already built. If a screen needs many calls, add one dedicated `/home` endpoint before reaching for GraphQL. AI answers will stream with SSE.

## English
Words: **extensible** (easy to add to without rewriting), **resilient** (recovers quickly from failure). Grammar: **past simple** (a finished action at a specific past time: "I built the server yesterday") vs **present perfect** (`have/has` + past participle: past action connected to now: "I have built three endpoints this week"; signal words: *already, yet, ever, never, just, so far, this week*).
✏️ Three sentences about today's topic, using both words, one in the past simple and one in the present perfect.

## Role lens · Team Lead
**How would you split work on "estimation + API styles" across 3 engineers, and what is the biggest delivery risk?**
✏️ Shape: *Split: ___ / ___ / ___. Risk: ___ because ___. Mitigation: ___.*

## Quiz and mistakes
- My Kaizen estimate was fully correct (3.47 peak QPS, 32.85 GB per year of text with replicas, 91.25 GB per year of photos).
- Units: capital **B** is bytes, lowercase **b** is bits. I wrote "kb" and "Gb" for storage: it should be KB and GB.
- I left out the bandwidth step (50,000 reads × 20 KB = 1 GB per day, about 12 KB/s).

## In my own words
✏️ Why is "what you fetch" more important than "how you encode it"?

## Next
**BE-03** (15 Oct): requests, responses, serialization, validation and status codes: it builds directly on this API design.